# 02 — ETL espaço-temporal e features

_Notebook do projeto **crime-sp-ml**. Reaproveita os módulos em `src/`._

In [1]:
import sys, pathlib
from IPython.display import Image, IFrame, display, Markdown
ROOT = pathlib.Path.cwd()
while not (ROOT/'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
FIG = ROOT/'reports'/'figures'; MAPS = ROOT/'reports'/'maps'; REP = ROOT/'reports'
print('Projeto:', ROOT.name)

Projeto: crime-sp-ml


Engenharia de atributos espaço-temporais (`src/features.py` + `src/data_loader.py`). A validação espacial é por *bounding box* do município (sem shapefile). A hora recebe **codificação cíclica**.

In [2]:
import data_loader as dl, features as ft
df = dl.load_clean()
print('geo validas:', f"{int(df['geo_valid'].sum()):,}",
      '| com hora exata:', f"{int(df['tem_hora_exata'].sum()):,}")

geo validas: 1,885,004 | com hora exata: 1,379,343


### Subset supervisionado (geo válida + HORA EXATA)

In [3]:
sup = dl.get_supervised_df(df)
sup[['lat','lon','hora','hora_sin','hora_cos','periodo','crime']].head()

,lat,lon,hora,hora_sin,hora_cos,periodo,crime
0,-23.555209,-46.635211,1,2.588190e-01,0.965926,madrugada,FURTO - OUTROS
1,-23.563815,-46.632484,21,-7.071068e-01,0.707107,noite,FURTO - OUTROS
2,-23.543624,-46.632520,12,1.224647e-16,-1.000000,tarde,FURTO - OUTROS
3,-23.557062,-46.629103,9,7.071068e-01,-0.707107,manha,FURTO - OUTROS
4,-23.557241,-46.632991,11,2.588190e-01,-0.965926,manha,FURTO - OUTROS


### Codificação cíclica da hora (sin/cos)

In [4]:
tab = sup[['hora','hora_norm','hora_sin','hora_cos']].drop_duplicates().sort_values('hora')
tab.head(24)

,hora,hora_norm,hora_sin,hora_cos
28,0,0.000000,0.000000e+00,1.000000e+00
0,1,0.043478,2.588190e-01,9.659258e-01
37,2,0.086957,5.000000e-01,8.660254e-01
73,3,0.130435,7.071068e-01,7.071068e-01
108,4,0.173913,8.660254e-01,5.000000e-01
25,5,0.217391,9.659258e-01,2.588190e-01
17,6,0.260870,1.000000e+00,6.123234e-17
16,7,0.304348,9.659258e-01,-2.588190e-01
31,8,0.347826,8.660254e-01,-5.000000e-01
3,9,0.391304,7.071068e-01,-7.071068e-01


### Trilha exploratória (período → hora aproximada, sinalizada)

In [5]:
exp = dl.get_exploratory_df(df)
print('linhas exploratorio:', f'{len(exp):,}')
print('com hora aproximada por periodo:', f"{int(exp['hora_aproximada'].sum()):,}")

linhas exploratorio: 1,805,961
com hora aproximada por periodo: 564,705


A matriz de clustering usa `lat, lon, hora_sin, hora_cos` escalonadas:

In [6]:
X, scaler, names = ft.build_cluster_matrix(exp.sample(5000, random_state=42))
print('shape:', X.shape, '| features:', names)

shape: (5000, 4) | features: ['lat', 'lon', 'hora_sin', 'hora_cos']
